# 02. 데이터 분포 분석

## 학습 목표
- Histogram과 KDE로 데이터 분포 이해
- Boxplot과 Violinplot으로 이상치 탐지
- 분포 특성에서 비즈니스 인사이트 도출

---

## 1. 왜 분포 분석이 중요한가?

**제조업 실무 사례:**
- 부품 치수 분포 → 불량률 예측
- 차량 연비 분포 → 규제 준수 여부 판단
- 이상치 탐지 → 공정 이상 조기 발견

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
mpg = sns.load_dataset('mpg')
print(f"전체 데이터: {len(mpg)}대")
mpg.describe()

## 2. Histogram: 도수 분포 파악

**비즈니스 질문:** 우리 차량의 연비는 정규분포를 따르는가?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram 생성
n, bins, patches = ax.hist(mpg['mpg'], bins=25, edgecolor='black', 
                           color='#3A86FF', alpha=0.7, linewidth=1.2)

# 평균선 추가
mean_mpg = mpg['mpg'].mean()
ax.axvline(mean_mpg, color='#FF006E', linestyle='--', linewidth=2.5, 
           label=f'평균: {mean_mpg:.1f} mpg')

# 중앙값 추가
median_mpg = mpg['mpg'].median()
ax.axvline(median_mpg, color='#FFBE0B', linestyle='--', linewidth=2.5,
           label=f'중앙값: {median_mpg:.1f} mpg')

ax.set_xlabel('연비 (mpg)', fontsize=12)
ax.set_ylabel('차량 대수', fontsize=12)
ax.set_title('자동차 연비 분포 (Histogram)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"평균: {mean_mpg:.2f} mpg")
print(f"중앙값: {median_mpg:.2f} mpg")
print(f"표준편차: {mpg['mpg'].std():.2f} mpg")

# 💡 인사이트: 평균 > 중앙값 → 우측 꼬리(고연비 차량) 존재, 정규분포 아님
# 💡 10~20mpg 구간에 차량 집중 → 저연비 차량 비중 높음

## 3. KDE (Kernel Density Estimation): 부드러운 분포 곡선

**비즈니스 질문:** 제조국별 연비 분포의 차이는?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# 제조국별 KDE 플롯
colors = {'usa': '#E63946', 'japan': '#06FFA5', 'europe': '#457B9D'}

for origin, color in colors.items():
    data = mpg[mpg['origin'] == origin]['mpg']
    data.plot.kde(ax=ax, color=color, linewidth=3, label=f'{origin} (평균: {data.mean():.1f})')

ax.set_xlabel('연비 (mpg)', fontsize=12)
ax.set_ylabel('밀도', fontsize=12)
ax.set_title('제조국별 연비 분포 비교 (KDE)', fontsize=14, fontweight='bold')
ax.legend(title='제조국', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(5, 50)

plt.tight_layout()
plt.show()

# 💡 인사이트: 일본 차량은 25-35mpg에 집중 (단봉형), 미국은 10-20mpg (저연비 편중)
# 💡 유럽은 넓게 분포 → 다양한 세그먼트 커버

## 4. Boxplot: 이상치와 사분위수

**비즈니스 질문:** 실린더 수별로 연비 편차가 얼마나 다른가?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Boxplot 생성
mpg_clean = mpg[mpg['cylinders'].isin([4, 6, 8])].copy()

box_data = [mpg_clean[mpg_clean['cylinders'] == cyl]['mpg'] for cyl in [4, 6, 8]]
bp = ax.boxplot(box_data, labels=['4기통', '6기통', '8기통'],
                patch_artist=True, notch=True, widths=0.6)

# 박스 색상 설정
colors = ['#06FFA5', '#FFBE0B', '#E63946']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# 중앙값선 강조
for median in bp['medians']:
    median.set(color='black', linewidth=2)

ax.set_xlabel('실린더 수', fontsize=12)
ax.set_ylabel('연비 (mpg)', fontsize=12)
ax.set_title('실린더 수별 연비 분포 (Boxplot)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 통계 요약
for cyl in [4, 6, 8]:
    data = mpg_clean[mpg_clean['cylinders'] == cyl]['mpg']
    q1, q3 = data.quantile([0.25, 0.75])
    iqr = q3 - q1
    print(f"{cyl}기통 - 중앙값: {data.median():.1f}, IQR: {iqr:.1f}, 이상치 개수: {len(data[(data < q1-1.5*iqr) | (data > q3+1.5*iqr)])}")

# 💡 인사이트: 4기통은 IQR이 좁음(품질 안정), 8기통은 중앙값이 낮지만 이상치(고효율 모델) 존재
# 💡 Notch(신뢰구간) 겹치지 않음 → 실린더별 연비 차이 통계적으로 유의미

## 5. Violinplot: 분포 형태까지 표현

**비즈니스 질문:** 연식에 따라 연비 분포가 어떻게 진화했나?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Violinplot 생성
sns.violinplot(data=mpg, x='model_year', y='mpg', ax=ax,
               palette='coolwarm', inner='quartile', cut=0)

ax.set_xlabel('연식', fontsize=12)
ax.set_ylabel('연비 (mpg)', fontsize=12)
ax.set_title('연식별 연비 분포 변화 (Violinplot)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 💡 인사이트: 73년 이전은 저연비에 집중(바이올린 하단 볼륨 큼), 78년 이후 고연비 영역 확대
# 💡 82년 바이올린이 가장 넓음 → 제품 라인업 다양화 시점

## 6. 복합 분포 분석: Histogram + KDE + Boxplot

**종합 분석: 차량 무게 분포**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# 상단: Histogram + KDE
axes[0].hist(mpg['weight'], bins=30, edgecolor='black', 
             color='#3A86FF', alpha=0.6, density=True, label='Histogram')

mpg['weight'].plot.kde(ax=axes[0], color='#FF006E', linewidth=3, label='KDE')

axes[0].set_xlabel('차량 무게 (lbs)', fontsize=12)
axes[0].set_ylabel('밀도', fontsize=12)
axes[0].set_title('차량 무게 분포 (Histogram + KDE)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# 하단: Boxplot (제조국별)
box_data_weight = [mpg[mpg['origin'] == o]['weight'] for o in ['usa', 'japan', 'europe']]
bp = axes[1].boxplot(box_data_weight, labels=['USA', 'Japan', 'Europe'],
                     patch_artist=True, vert=False, widths=0.6)

for patch, color in zip(bp['boxes'], ['#E63946', '#06FFA5', '#457B9D']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_xlabel('차량 무게 (lbs)', fontsize=12)
axes[1].set_ylabel('제조국', fontsize=12)
axes[1].set_title('제조국별 차량 무게 비교 (Boxplot)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# 💡 인사이트: 미국 차량 중앙값 3,500lbs vs 일본 2,200lbs → 약 1,300lbs(60%) 차이
# 💡 일본 차량의 낮은 무게가 높은 연비의 핵심 요인

---

## 📊 핵심 요약

### 차트 선택 기준 (30%)
| 목적 | 도구 | 코드 패턴 |
|------|------|----------|
| 도수 분포 | Histogram | `ax.hist(data, bins=n)` |
| 부드러운 밀도 | KDE | `data.plot.kde(ax=ax)` |
| 이상치 탐지 | Boxplot | `ax.boxplot(data)` |
| 분포 형태 비교 | Violinplot | `sns.violinplot()` |

### 도출된 인사이트 (70%)
1. **분포 왜도**: 연비 데이터는 우측 꼬리 → 평균보다 중앙값이 대표성 높음
2. **국가별 전략**: 일본(경량화), 미국(대형화), 유럽(다변화)
3. **시계열 변화**: 1975년 이후 고연비 영역 확대 → 규제 효과
4. **이상치 의미**: 8기통 중 고연비 차량 존재 → 기술 혁신 사례

### 실무 의사결정
- **품질 관리**: IQR 모니터링으로 공정 안정성 평가
- **제품 기획**: 무게 2,500lbs 이하 목표 설정 → 연비 30mpg 달성 가능
- **벤치마킹**: 일본 경량화 기술 도입 검토